<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1 | Represent Words with Embeddings</h3>

In [1]:
import os

# Set the backend that keras uses before importing keras.
os.environ["KERAS_BACKEND"] = "tensorflow"

# Suppress warnings about GPU support.
# "2" hides INFO and WARNING messages but still shows ERROR messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=ALL, 1=INFO, 2=WARNING, 3=ERROR

import tensorflow as tf
import tensorflow_datasets as tfds
import keras
import numpy as np

Load the data from the AG News dataset.

In [2]:
dataset = tfds.load("ag_news_subset")
ds_train = dataset["train"]
ds_test = dataset["test"]

<h4 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.1 | What's an embedding?</h4>

Representing words using lower-dimensional dense vectors that reflect the semantic meaning of the word i.e. <span style="text-decoration:underline;">a way to reduce the dimensionality of a word vector</span>.

The classifier neural network will consist of a number of layers.
The model classifies the averaged meaning. This architecture is a simple bag-of-words-style model:

- It uses token embeddings.
- It averages them.
- It loses word order.

First, we use **Input** to initialize a keras tensor-like object with shape=(1,) which means the expected input will be in batches of 1-dimensional vectors.

- [**TextVectorization layer**](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization)<br/>
Takes a string as input and after standardizing the text
(lowercasing, stripping punctuation etc) it tokenizes it (splits the text into individual tokens-words). Finally, it replaces each token with the corresponding index from the internal vocabulary.
- [**Embedding layer**](https://keras.io/api/layers/core_layers/embedding)<br/>
Takes as input n word indices, and converts each index to a dense vector of length 100. So, for an input of shape (n,) the output is (n, 100).<br/>
These vector representations <span style="text-decoration:underline;">do not have any semantic meaning!</span>
- [**Aggregation layer**](https://keras.io/api/layers/core_layers/lambda)<br/>
Calculates the arithmetic mean of the word vectors. For an input matrix of shape (n, 100) the output is (100,)
- [**Dense layer**](https://keras.io/api/layers/core_layers/dense)<br/>
Applies a linear transformation (weights/biases) to make a final prediction.

In [4]:
# Specify a reasonable vocabulary size, ignoring less-frequently used words.
vocab_size = 30000
batch_size = 128

vectorizer = keras.layers.TextVectorization(max_tokens=vocab_size, output_mode="int")
model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        keras.layers.Embedding(vocab_size, 100),
        keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1)),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, None)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 100)      │     3,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           404 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,000,404 (11.45 MB)

 Trainable params: 3,000,404 (11.45 MB)

 Non-trainable params: 0 (0.00 B)

In the summary above, the first **None** corresponds to the batch size and the second **None** to the length of the token sequence. e.g. the sentence "I love to embed strings" has a sequence length of 5.

Next, we train the NN:

In [5]:
def extract_text(x):
    return x["title"] + " " + x["description"]


def tupelize(x):
    return (extract_text(x), x["label"])


print("Training vectorizer")

# Adapt the vectorizer to the training text. Use only
# a subset of 500, to speed up the process.
vectorizer.adapt(ds_train.take(500).map(extract_text))
# Compile the model
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
# Train the model
model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

Training vectorizer


c:\Dev\Python\JupyterNotebooks\Tutorials\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


938/938 ━━━━━━━━━━━━━━━━━━━━ 30s 31ms/step - accuracy: 0.7963 - loss: 0.6913 - val_accuracy: 0.8714 - val_loss: 0.4239


<h4 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">1.2 | Dealing with variable sequence sizes</h4>

The number of tokens/words in each sentence is different:

In [6]:
print(vectorizer("Hello, world!"))
print(vectorizer("I am glad to meet you!"))

tf.Tensor([ 1 45], shape=(2,), dtype=int64)
tf.Tensor([ 112 1271    1    3 1747  158], shape=(6,), dtype=int64)


Since the vectorizer is applied to sequences of variable length, it fills unused elements with the PAD token (a zero), so that the output tensor is of rectangular shape.

In [7]:
vectorizer(["Hello, world!", "I am glad to meet you!"])

<tf.Tensor: shape=(2, 6), dtype=int64, numpy=
array([[   1,   45,    0,    0,    0,    0],
       [ 112, 1271,    1,    3, 1747,  158]])>

In [8]:
# layers[1] is the Embedding layer to which we pass the vectorizer. Input is not a layer
model.layers[1](vectorizer(["Hello, world!", "I am glad to meet you!"])).numpy()

array([[[-0.04757385, -0.00179866, -0.09475154, ..., -0.02992923,
          0.02674875,  0.11258027],
        [ 0.17626666, -0.22254182, -0.26720613, ..., -0.30451986,
          0.16968402,  0.03898042],
        [ 0.02283046, -0.0083607 ,  0.00079246, ..., -0.02087448,
          0.01667224, -0.00441453],
        [ 0.02283046, -0.0083607 ,  0.00079246, ..., -0.02087448,
          0.01667224, -0.00441453],
        [ 0.02283046, -0.0083607 ,  0.00079246, ..., -0.02087448,
          0.01667224, -0.00441453],
        [ 0.02283046, -0.0083607 ,  0.00079246, ..., -0.02087448,
          0.01667224, -0.00441453]],

       [[ 0.08659261,  0.01449614, -0.2564625 , ..., -0.32854375,
          0.3286786 ,  0.22586732],
        [-0.01090629,  0.13037148, -0.02594715, ..., -0.07476472,
          0.09183814,  0.13183708],
        [-0.04757385, -0.00179866, -0.09475154, ..., -0.02992923,
          0.02674875,  0.11258027],
        [-0.10570256,  0.17308149,  0.16453232, ...,  0.13506481,
         -0.08

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">2 | Semantic embeddings: Word2Vec</h3>

In order to obtain a vector representation of a words such that similar words/synonyms correspond to vectors close to each other, we must pretrain our model on a large collection of text using a technique such as [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). The two main architectures are:

- **Continuous bag-of-words (CBoW)**<br/>
The model is trained to predict a word from the surrounding context.
- **Continuous skip-gram**<br/>
The model uses an input word to predict the surrounding window of context words.


In [10]:
import gensim.downloader as api

# Download/load the pretrained Google News Word2Vec model.
# To uninstall the package : pip uninstall gensim
# To uninstall the entire cache: Remove-Item -Recurse -Force "C:\Users\nkmil\gensim-data"
w2v = api.load("word2vec-google-news-300")

# Find and display words most similar to "neural"
for w, p in w2v.most_similar("woman"):
    print(f"{w} -> {p}")

man -> 0.7664012908935547
girl -> 0.7494640946388245
teenage_girl -> 0.7336829304695129
teenager -> 0.6317085027694702
lady -> 0.6288785934448242
teenaged_girl -> 0.6141784191131592
mother -> 0.6076306104660034
policewoman -> 0.6069462299346924
boy -> 0.5975907444953918
Woman -> 0.5770983099937439


In [11]:
# Extract the vector embedding from the word. The embedding has
# 300 components; here we only show the first 20.
w2v["play"][:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

In [ ]:
# Find a word as close as possible to king and woman AND as far as possible from the word man.
w2v.most_similar(positive=["king", "woman"], negative=["man"])[0]

('queen', 0.7118193507194519)

In [13]:
# Get the vector corresponding to KING-MAN+WOMAN
qvec = w2v["king"] - w2v["man"] + w2v["woman"]

# Find the index of the closest embedding vector, excluding input words
d = np.sum((w2v.vectors - qvec) ** 2, axis=1)
exclude = {w2v.key_to_index[w] for w in ["king", "man", "woman"]}
d[list(exclude)] = np.inf
min_idx = np.argmin(d)

# Find the corresponding word
w2v.index_to_key[min_idx]

'queen'

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">3 | Using pretrained embeddings in Keras</h3>

<h4 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">3.1 | Using tokenizer vocabulary</h4>

In [16]:
embed_size = len(w2v.get_vector("hello"))
print(f"Embedding size: {embed_size}")

vocab = vectorizer.get_vocabulary()
W = np.zeros((vocab_size, embed_size))
print(f"Populating matrix. This will take some time...", end="")

found, not_found = 0, 0

for i, w in enumerate(vocab):
    try:
        W[i] = w2v.get_vector(w)
        found += 1
    except KeyError:
        not_found += 1

print(f"Done, found {found} words, {not_found} words missing")

Embedding size: 300
Populating matrix. This will take some time...Done, found 4551 words, 784 words missing


For the 784 words not found in the Word2Vec vocabulary, we can leave them as 0 or create a random vector.

Now, define an embedding layer with pretrained weights:

In [17]:
emb = keras.layers.Embedding(vocab_size, embed_size, weights=[W], trainable=False)
model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        emb,
        keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1)),
        keras.layers.Dense(4, activation="softmax"),
    ]
)

In [19]:
# Train this model
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7818 - loss: 1.1240 - val_accuracy: 0.8170 - val_loss: 0.9457


<h4 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">3.2 | Using embedding vocabulary</h4>

Note that the vocabularies used in TextVectorization and Embedding are different. Instead, we can load our dataset with the pretrained vocabulary from Word2Vec model.

In [20]:
vocab = list(w2v.key_to_index.keys())
vectorizer = keras.layers.TextVectorization()
vectorizer.set_vocabulary(vocab)

Next, build the embedding weight matrix from the Word2Vec vectors.

In [21]:
embed_size = w2v.vector_size
vocab_size_w2v = len(vocab) + 2  # for padding and unknown tokens
W = np.zeros((vocab_size_w2v, embed_size))

for i, word in enumerate(vocab):
    W[i + 2] = w2v[word]

emb_w2v = keras.layers.Embedding(
    vocab_size_w2v,
    embed_size,
    weights=[W],
    trainable=False,
)

model = keras.Sequential(
    [
        keras.Input(shape=(1,), dtype=tf.string),
        vectorizer,
        emb_w2v,
        keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1)),
        keras.layers.Dense(4, activation="softmax"),
    ]
)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    ds_train.map(tupelize).batch(batch_size),
    validation_data=ds_test.map(tupelize).batch(batch_size),
)

938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8144 - loss: 1.0504 - val_accuracy: 0.8421 - val_loss: 0.8406


<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">4 | Training your own embeddings</h3>

Check the official [TensorFlow tutorial on training Word2Vec model](https://www.tensorflow.org/tutorials/text/word2vec)

<h3 style="font-weight:600;color:white;background:#4f4a9e;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 10px 10px 10px;text-transform:uppercase;">5 | Contextual embeddings</h3>